# Evidex: calibrar el umbral del modelo

No entrena de nuevo: toma el modelo ya entrenado y ajusta **desde qué puntaje se da la alerta**, usando fotos
originales que el modelo nunca vio (partición *validation* de TGIF). Después mide con la partición *testing*.

Antes de empezar:
1. Suba el `evidex_modelo.zip` de la vuelta 3 como *Dataset* (Create → New Dataset, por ejemplo «evidex-modelo-v3»)
   y agréguelo con **+ Add Input**.
2. **Settings → Internet:** activado. No necesita GPU (Accelerator: None).

Después: **Save Version → Save & Run All (Commit)**. Tarda menos de 1 hora. Al final descargue
`evidex_modelo_calibrado.zip` desde **Output** y copie los 3 archivos a `veritas\modelos\`.

In [ ]:
# 1. Configuración
import os
FPR = float(os.environ.get("EVX_FPR", 0.015))          # meta: como máximo 1,5 % de originales con alarma
N_ORIG_VAL = int(os.environ.get("EVX_N_ORIG_VAL", 1200))   # originales para calibrar (más = umbral más estable)
N_EDIT_VAL = int(os.environ.get("EVX_N_EDIT_VAL", 200))    # editadas por tipo, para ver cuánto se detecta
N_PRUEBA = int(os.environ.get("EVX_N_PRUEBA", 300))
BASE = "/kaggle/temp" if os.path.isdir("/kaggle") else "."
SALIDA = "/kaggle/working/modelos" if os.path.isdir("/kaggle") else "modelos_calibrado"

In [ ]:
# 2. Preparar
import subprocess, sys, glob, shutil
def sh(cmd):
    print("$", cmd, flush=True)
    if subprocess.run(cmd, shell=True).returncode:
        raise SystemExit(f"Falló: {cmd}")
sh(f"{sys.executable} -m pip install -q onnxruntime")
from pathlib import Path
Path("training").mkdir(exist_ok=True)
ARCHIVOS = {
 "datos.py": "\"\"\"Datos para entrenar el detector de ediciones con IA.\n\nTodo pasa por un manifiesto CSV (una fila por imagen), para saber siempre de dónde salió cada foto y con qué\nlicencia:  ruta,etiqueta,mascara,conjunto,licencia\n  etiqueta: 1 = editada/generada con IA, 0 = original\n  mascara: PNG blanco donde se cambió (vacío = sin máscara; para originales se usa todo negro)\n\nConjuntos soportados de entrada:\n  - TGIF / TGIF2 (carpetas orig/, sd2-sp/, sdxl-fr/, ... y masks/), CC BY 4.0 / CC BY-SA 4.0.\n  - Fotos propias (carpeta con originales/ y editadas/, y opcionalmente mascaras/ con el mismo nombre).\n    Use solo fotos suyas o con permiso escrito; los datos reales de BCI requieren autorización de BCI.\n\nLas licencias NO comerciales quedan bloqueadas: el script se niega a usarlas (ver LICENCIAS_BLOQUEADAS).\n\"\"\"\nfrom __future__ import annotations\n\nimport csv\nimport io\nimport random\nimport re\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\nIMAGES = {\".jpg\", \".jpeg\", \".png\", \".webp\"}\nLICENCIAS_BLOQUEADAS = (\"nc\", \"non-commercial\", \"noncommercial\", \"no comercial\", \"research only\", \"desconocida\")\nCAMPOS = [\"ruta\", \"etiqueta\", \"mascara\", \"conjunto\", \"licencia\"]\n\n# subcarpeta de TGIF -> licencia (README de IDLabMedia/tgif-dataset)\nTGIF_LICENCIA = \"CC BY 4.0 (TGIF) / CC BY-SA 4.0 (TGIF2); imágenes base MS-COCO\"\nTGIF_EVITAR = (\"flux\",)          # FLUX.1 dev tiene licencia no comercial: esas imágenes no se usan\n\n\ndef licencia_permitida(lic: str) -> bool:\n    l = (lic or \"desconocida\").lower()\n    return not any(b in l for b in LICENCIAS_BLOQUEADAS)\n\n\ndef _mascara_tgif(img: Path, masks: Path) -> str:\n    # «368961_mask_bbox.png_ps_mask.png_sd2_0.png» -> masks/<split>/<clase>/368961_mask_bbox.png\n    m = re.match(r\"(.+?_mask_(?:bbox|segm)(?:_512)?\\.png)\", img.name)\n    if not m:\n        return \"\"\n    rel = img.parent.relative_to(img.parents[2])           # <split>/<clase>\n    cand = masks / rel / m.group(1)\n    return str(cand) if cand.exists() else \"\"\n\n\ndef manifiesto_tgif(raiz: Path) -> list[dict]:\n    \"\"\"raiz/orig/<split>/<clase>/*.png, raiz/<metodo>/<split>/<clase>/*.png, raiz/masks/...\"\"\"\n    filas, masks = [], raiz / \"masks\"\n    for carpeta in sorted(p for p in raiz.iterdir() if p.is_dir() and p.name != \"masks\"):\n        if any(x in carpeta.name.lower() for x in TGIF_EVITAR):\n            continue\n        original = carpeta.name == \"orig\"\n        # «-fr» = toda la imagen pasó por la IA (regenerada): la máscara es la imagen completa, no solo la zona\n        completa = carpeta.name.lower().endswith(\"-fr\")\n        for img in sorted(p for p in carpeta.rglob(\"*\") if p.suffix.lower() in IMAGES):\n            mask = \"\" if original or completa else _mascara_tgif(img, masks)\n            if not original and not completa and not mask:\n                continue                                     # editada sin máscara: no sirve para localizar\n            filas.append({\"ruta\": str(img), \"etiqueta\": 0 if original else 1, \"mascara\": mask,\n                          \"conjunto\": f\"TGIF/{carpeta.name}\", \"licencia\": TGIF_LICENCIA})\n    return filas\n\n\ndef manifiesto_propias(raiz: Path, licencia: str = \"propia (fotos de Zelekpress con permiso)\") -> list[dict]:\n    filas = []\n    for sub, et in ((\"originales\", 0), (\"editadas\", 1)):\n        for img in sorted(p for p in (raiz / sub).rglob(\"*\") if p.suffix.lower() in IMAGES):\n            mask = raiz / \"mascaras\" / (img.stem + \".png\")\n            filas.append({\"ruta\": str(img), \"etiqueta\": et, \"mascara\": str(mask) if mask.exists() else \"\",\n                          \"conjunto\": f\"propias/{sub}\", \"licencia\": licencia})\n    return filas\n\n\ndef guardar(filas: list[dict], destino: Path) -> None:\n    malas = sorted({f[\"licencia\"] for f in filas if not licencia_permitida(f[\"licencia\"])})\n    if malas:\n        raise SystemExit(f\"Licencias no permitidas para uso comercial: {malas}\")\n    with destino.open(\"w\", newline=\"\", encoding=\"utf-8\") as fh:\n        w = csv.DictWriter(fh, fieldnames=CAMPOS)\n        w.writeheader()\n        w.writerows(filas)\n\n\ndef leer(manifiesto: Path) -> list[dict]:\n    with manifiesto.open(encoding=\"utf-8\") as fh:\n        filas = list(csv.DictReader(fh))\n    for f in filas:\n        if not licencia_permitida(f[\"licencia\"]):\n            raise SystemExit(f\"{f['ruta']}: licencia no permitida ({f['licencia']})\")\n        f[\"etiqueta\"] = int(f[\"etiqueta\"])\n    return filas\n\n\n# ---- lo que le pasa a una foto en el camino (WhatsApp, capturas, reenvíos) -------------------------------\ndef como_whatsapp(img: Image.Image, mask: Image.Image, rng: random.Random) -> tuple[Image.Image, Image.Image]:\n    \"\"\"Reducción + recompresión JPEG (a veces doble) + recorte leve. Se aplica IGUAL a originales y editadas,\n    para que el modelo no aprenda «comprimida = editada».\"\"\"\n    if rng.random() < .3:                                   # recorte leve, como al reencuadrar en la galería\n        w, h = img.size\n        dx, dy = int(w * rng.uniform(0, .08)), int(h * rng.uniform(0, .08))\n        box = (dx, dy, w - int(w * rng.uniform(0, .08)), h - int(h * rng.uniform(0, .08)))\n        img, mask = img.crop(box), mask.crop(box)\n    lado = rng.choice([640, 800, 1024, 1280, 1600, 2048])\n    if max(img.size) > lado:\n        f = lado / max(img.size)\n        nuevo = (max(1, round(img.width * f)), max(1, round(img.height * f)))\n        img, mask = img.resize(nuevo, Image.LANCZOS), mask.resize(nuevo, Image.NEAREST)\n    for _ in range(rng.choice([1, 1, 2])):                  # a veces se reenvía: doble compresión\n        buf = io.BytesIO()\n        img.save(buf, \"JPEG\", quality=rng.randint(50, 92))\n        img = Image.open(io.BytesIO(buf.getvalue())).convert(\"RGB\")\n    return img, mask\n\n\ndef a_tensor(img: Image.Image, lado: int) -> np.ndarray:\n    \"\"\"Igual que en evidex/forensics/learned.py: RGB, lado x lado, 0..1, normalizado ImageNet, CHW.\"\"\"\n    x = np.asarray(img.convert(\"RGB\").resize((lado, lado), Image.BILINEAR), dtype=np.float32) / 255.0\n    x = (x - np.array([0.485, 0.456, 0.406], np.float32)) / np.array([0.229, 0.224, 0.225], np.float32)\n    return x.transpose(2, 0, 1)\n\n\n# ---- mosaico: la foto se analiza en pedazos a tamaño real, sin achicarla a 512 ----------------------------\n# (la misma lógica está en evidex/forensics/learned.py; tests/test_learned_model.py revisa que den lo mismo)\nMAX_LADO = 2048          # fotos más grandes se reducen a esto antes del mosaico (tiempo de análisis acotado)\n\n\ndef posiciones(n: int, lado: int, paso: int) -> list[int]:\n    if n <= lado:\n        return [0]\n    pos = list(range(0, n - lado, paso))\n    return pos + [n - lado]\n\n\ndef preparar(img: Image.Image, max_lado: int = MAX_LADO) -> Image.Image:\n    img = img.convert(\"RGB\")\n    if max(img.size) > max_lado:\n        f = max_lado / max(img.size)\n        img = img.resize((max(1, round(img.width * f)), max(1, round(img.height * f))), Image.LANCZOS)\n    return img\n\n\ndef normalizar(img: Image.Image) -> np.ndarray:\n    x = np.asarray(img, dtype=np.float32) / 255.0\n    return ((x - np.array([0.485, 0.456, 0.406], np.float32)) / np.array([0.229, 0.224, 0.225], np.float32))\n\n\ndef mosaico(img: Image.Image, lado: int, paso: int):\n    \"\"\"Pedazos lado x lado (CHW normalizados) y su posición. Si la foto es más chica que `lado`, se rellena\n    con negro (igual al entrenar y al analizar).\"\"\"\n    w, h = img.size\n    W, H = max(w, lado), max(h, lado)\n    x = np.zeros((H, W, 3), np.float32)\n    x[:] = normalizar(Image.new(\"RGB\", (1, 1)))[0, 0]          # negro normalizado\n    x[:h, :w] = normalizar(img)\n    tiles, pos = [], []\n    for yy in posiciones(H, lado, paso):\n        for xx in posiciones(W, lado, paso):\n            tiles.append(x[yy:yy + lado, xx:xx + lado].transpose(2, 0, 1))\n            pos.append((xx, yy))\n    return np.stack(tiles).astype(np.float32), pos, (w, h), (W, H)\n\n\ndef puntuar_mosaico(correr, img: Image.Image, lado: int, paso: int, lote: int = 8):\n    \"\"\"`correr(lote_np) -> (mapas N x1xh xw, puntajes N)`. Devuelve (puntaje de la foto, mapa de la foto\n    a 1/escala, escala). Puntaje = el del pedazo más sospechoso.\"\"\"\n    tiles, pos, (w, h), (W, H) = mosaico(img, lado, paso)\n    mapas, puntajes = [], []\n    for i in range(0, len(tiles), lote):\n        m, s = correr(tiles[i:i + lote])\n        mapas.append(np.asarray(m)); puntajes.append(np.asarray(s).reshape(-1))\n    mapas, puntajes = np.concatenate(mapas)[:, 0], np.concatenate(puntajes)\n    esc = lado // mapas.shape[-1]\n    acc = np.zeros((H // esc + 1, W // esc + 1), np.float32)\n    cnt = np.zeros_like(acc)\n    for (xx, yy), m in zip(pos, mapas):\n        acc[yy // esc:yy // esc + m.shape[0], xx // esc:xx // esc + m.shape[1]] += m\n        cnt[yy // esc:yy // esc + m.shape[0], xx // esc:xx // esc + m.shape[1]] += 1\n    mapa = (acc / np.maximum(cnt, 1))[:max(1, h // esc), :max(1, w // esc)]\n    return float(puntajes.max()), mapa, esc\n\n\ndef degradar_fijo(img: Image.Image) -> Image.Image:\n    \"\"\"Lo mismo que el banco de pruebas: lado mayor 1600, JPEG 70 (para validar siempre igual).\"\"\"\n    img = img.convert(\"RGB\")\n    img.thumbnail((1600, 1600), Image.LANCZOS)\n    buf = io.BytesIO()\n    img.save(buf, \"JPEG\", quality=70)\n    return Image.open(io.BytesIO(buf.getvalue())).convert(\"RGB\")\n\n\nclass Recortes:\n    \"\"\"Dataset de entrenamiento en recortes lado x lado a tamaño real (sin achicar la foto entera).\n    En las editadas, la mayoría de los recortes caen sobre la zona cambiada. La etiqueta es del RECORTE:\n    1 si contiene parte de la zona editada.\"\"\"\n\n    def __init__(self, filas: list[dict], lado: int = 512, semilla: int = 0, sobre_zona: float = .6):\n        self.filas, self.lado, self.semilla, self.sobre_zona = filas, lado, semilla, sobre_zona\n        self.epoca = 0\n\n    def __len__(self):\n        return len(self.filas)\n\n    def __getitem__(self, i):\n        f, L = self.filas[i], self.lado\n        rng = random.Random(hash((self.semilla, self.epoca, i)))\n        img = Image.open(f[\"ruta\"]).convert(\"RGB\")\n        if f[\"mascara\"]:\n            mask = Image.open(f[\"mascara\"]).convert(\"L\").resize(img.size, Image.NEAREST)\n        else:\n            mask = Image.new(\"L\", img.size, 255 if f[\"etiqueta\"] else 0)\n        img, mask = como_whatsapp(img, mask, rng)\n        if max(img.size) > MAX_LADO:                         # como_whatsapp ya deja <= 2048; por si acaso\n            img = preparar(img)\n            mask = mask.resize(img.size, Image.NEAREST)\n        if rng.random() < .5:\n            img, mask = img.transpose(Image.FLIP_LEFT_RIGHT), mask.transpose(Image.FLIP_LEFT_RIGHT)\n        w, h = img.size\n        m = np.asarray(mask) > 127\n        if f[\"etiqueta\"] and m.any() and not m.all() and rng.random() < self.sobre_zona:\n            ys, xs = np.nonzero(m)\n            k = rng.randrange(len(xs))\n            cx, cy = xs[k] + rng.randint(-L // 4, L // 4), ys[k] + rng.randint(-L // 4, L // 4)\n            x0, y0 = cx - L // 2, cy - L // 2\n        else:\n            x0, y0 = rng.randint(0, max(0, w - L)), rng.randint(0, max(0, h - L))\n        x0, y0 = min(max(0, x0), max(0, w - L)), min(max(0, y0), max(0, h - L))\n        tiles, _, _, _ = mosaico(img.crop((x0, y0, min(w, x0 + L), min(h, y0 + L))), L, L)\n        mm = np.zeros((L, L), np.float32)\n        mc = m[y0:y0 + L, x0:x0 + L]\n        mm[:mc.shape[0], :mc.shape[1]] = mc\n        return tiles[0], mm[None], np.float32(mm.mean() > .001)\n\n\nclass Conjunto:\n    \"\"\"Dataset compatible con torch.utils.data (sin depender de torch aquí).\"\"\"\n\n    def __init__(self, filas: list[dict], lado: int = 512, aumentar: bool = True, semilla: int = 0):\n        self.filas, self.lado, self.aumentar, self.semilla = filas, lado, aumentar, semilla\n        self.epoca = 0\n\n    def __len__(self):\n        return len(self.filas)\n\n    def __getitem__(self, i):\n        f = self.filas[i]\n        rng = random.Random(hash((self.semilla, self.epoca if self.aumentar else 0, i)))\n        img = Image.open(f[\"ruta\"]).convert(\"RGB\")\n        if f[\"mascara\"]:\n            mask = Image.open(f[\"mascara\"]).convert(\"L\").resize(img.size, Image.NEAREST)\n        else:\n            mask = Image.new(\"L\", img.size, 255 if (f[\"etiqueta\"] and not f[\"mascara\"]) else 0)\n        img, mask = como_whatsapp(img, mask, rng)\n        if self.aumentar and rng.random() < .5:\n            img, mask = img.transpose(Image.FLIP_LEFT_RIGHT), mask.transpose(Image.FLIP_LEFT_RIGHT)\n        m = (np.asarray(mask.resize((self.lado, self.lado), Image.NEAREST)) > 127).astype(np.float32)\n        return a_tensor(img, self.lado), m[None], np.float32(f[\"etiqueta\"])\n\n\ndef dividir(filas: list[dict], val: float = .15, semilla: int = 0) -> tuple[list[dict], list[dict]]:\n    \"\"\"Separa por imagen base (id COCO / nombre), para que la original y su edición no queden una en\n    entrenamiento y otra en validación (eso inflaría las métricas).\"\"\"\n    def base(f):\n        return re.match(r\"(\\d+|[^_.]+)\", Path(f[\"ruta\"]).name).group(1)\n    ids = sorted({base(f) for f in filas})\n    random.Random(semilla).shuffle(ids)\n    v = set(ids[:max(1, int(len(ids) * val))])\n    return [f for f in filas if base(f) not in v], [f for f in filas if base(f) in v]\n\n\nif __name__ == \"__main__\":\n    import argparse\n    ap = argparse.ArgumentParser(description=\"Crea el manifiesto CSV de entrenamiento.\")\n    ap.add_argument(\"--tgif\", type=Path, action=\"append\", default=[], help=\"carpeta raíz de TGIF/TGIF2\")\n    ap.add_argument(\"--propias\", type=Path, action=\"append\", default=[], help=\"carpeta con originales/ y editadas/\")\n    ap.add_argument(\"--salida\", type=Path, default=Path(\"manifiesto.csv\"))\n    a = ap.parse_args()\n    filas = [f for r in a.tgif for f in manifiesto_tgif(r)] + [f for r in a.propias for f in manifiesto_propias(r)]\n    guardar(filas, a.salida)\n    print(f\"{len(filas)} imágenes ({sum(f['etiqueta'] for f in filas)} editadas) -> {a.salida}\")\n",
 "descargar_tgif.py": "\"\"\"Descarga una parte de TGIF (CC BY 4.0) sin bajar los archivos completos (algunos pesan 14 GB).\n\nLee el .tar.gz a medida que llega y se detiene al tener N imágenes de cada carpeta.\n  python training/descargar_tgif.py --destino datos/tgif --n 3000 --particion training\n  python training/descargar_tgif.py --destino datos/tgif --n 500  --particion validation\n\nOrigen: https://github.com/IDLabMedia/tgif-dataset (enlaces públicos de Nextcloud de IDLab, imec).\nCite el artículo de TGIF y respete CC BY 4.0 (atribución). No se bajan las variantes FLUX (licencia no comercial).\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport tarfile\nimport urllib.request\nfrom pathlib import Path\n\nBASE = \"https://cloud.ilabt.imec.be/public.php/dav/files/xEeAzrY7ES9KA8o\"\nCARPETAS = (\"orig\", \"sd2-sp\", \"sd2-fr\", \"sdxl-fr\", \"ps-sp\")\n\n\ndef bajar(carpeta: str, particion: str, destino: Path, n: int) -> int:\n    url = f\"{BASE}/{carpeta}/{carpeta}_{particion}.tar.gz\"\n    req = urllib.request.Request(url, headers={\"X-Requested-With\": \"XMLHttpRequest\"})\n    k = 0\n    with urllib.request.urlopen(req, timeout=120) as r, tarfile.open(fileobj=r, mode=\"r|gz\") as t:\n        for m in t:\n            if not m.isfile():\n                continue\n            t.extract(m, destino / carpeta, filter=\"data\")\n            k += 1\n            if n and k >= n:\n                break\n    return k\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument(\"--destino\", type=Path, required=True)\n    ap.add_argument(\"--n\", type=int, default=2000, help=\"imágenes por carpeta (0 = todas)\")\n    ap.add_argument(\"--particion\", choices=(\"training\", \"validation\", \"testing\"), default=\"training\")\n    ap.add_argument(\"--carpetas\", default=\",\".join(CARPETAS))\n    a = ap.parse_args()\n    for c in a.carpetas.split(\",\"):\n        print(c, bajar(c, a.particion, a.destino, a.n), flush=True)\n    print(\"masks\", bajar(\"masks\", a.particion, a.destino, 0))      # las máscaras pesan poco: todas\n\n\nif __name__ == \"__main__\":\n    main()\n",
 "calibrar.py": "\"\"\"Calibra el umbral de un modelo ya entrenado, sin volver a entrenarlo.\n\nEl umbral decide desde qué puntaje Evidex da la alerta. Se fija con fotos ORIGINALES que el modelo nunca vio\n(partición «validation» de TGIF), para que como máximo el --fpr de ellas dé alarma, y después se mide con la\npartición «testing» (otras fotos, también nunca vistas). Todas pasan antes por WhatsApp (1600 px, JPEG 70) y se\nanalizan en mosaico, igual que en Evidex.\n\n  python training/calibrar.py --modelo modelos/ --validacion datos_val --prueba datos_prueba --fpr 0.015 --salida modelos_cal/\n\nDeja en --salida el mismo .onnx (sin cambios: su sha256 sigue valiendo) y la ficha con el umbral nuevo.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport glob\nimport hashlib\nimport json\nimport shutil\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom datos import degradar_fijo, preparar, puntuar_mosaico  # noqa: E402\n\nTIPOS = {\"orig\": \"originales (falsas alarmas)\", \"sd2-sp\": \"zona editada SD2\", \"sd2-fr\": \"regenerada SD2\",\n         \"sdxl-fr\": \"regenerada SDXL\", \"ps-sp\": \"zona editada Photoshop\"}\n\n\ndef sesion(onnx: Path):\n    import onnxruntime as ort\n    opts = ort.SessionOptions()\n    return ort.InferenceSession(str(onnx), opts, providers=[\"CPUExecutionProvider\"])\n\n\ndef puntajes(sess, ficha, carpeta: Path) -> np.ndarray:\n    correr = lambda t: sess.run([\"mapa\", \"puntaje\"], {\"imagen\": t})      # noqa: E731\n    out = []\n    for p in sorted(glob.glob(str(carpeta / \"**\" / \"*.png\"), recursive=True)):\n        img = preparar(degradar_fijo(Image.open(p)), ficha.get(\"max_lado\", 2048))\n        out.append(puntuar_mosaico(correr, img, ficha[\"lado\"], ficha.get(\"paso\", ficha[\"lado\"] * 3 // 4))[0])\n    return np.array(out)\n\n\ndef umbral_para(orig: np.ndarray, fpr: float) -> float:\n    \"\"\"El menor umbral con el que, como máximo, el `fpr` de las originales da alarma.\"\"\"\n    s = np.sort(orig)\n    k = int(np.floor(len(s) * (1 - fpr)))\n    return float(s[min(k, len(s) - 1)]) + 1e-7\n\n\ndef tabla(res: dict, u: float) -> dict:\n    return {TIPOS[c]: f\"{int((v >= u).sum())} de {len(v)} ({100 * (v >= u).mean():.1f} %)\"\n            for c, v in res.items() if len(v)}\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument(\"--modelo\", type=Path, required=True, help=\"carpeta con evidex_ia.onnx y evidex_ia.json\")\n    ap.add_argument(\"--validacion\", type=Path, required=True, help=\"TGIF partición validation (orig/ y editadas)\")\n    ap.add_argument(\"--prueba\", type=Path, help=\"TGIF partición testing, para medir al final\")\n    ap.add_argument(\"--fpr\", type=float, default=.015, help=\"falsas alarmas máximas en originales\")\n    ap.add_argument(\"--salida\", type=Path, required=True)\n    a = ap.parse_args(argv)\n\n    ficha = json.loads((a.modelo / \"evidex_ia.json\").read_text(encoding=\"utf-8\"))\n    onnx = a.modelo / \"evidex_ia.onnx\"\n    if hashlib.sha256(onnx.read_bytes()).hexdigest() != ficha[\"sha256_onnx\"]:\n        raise SystemExit(\"El .onnx no coincide con su ficha (sha256): no se calibra un archivo cambiado.\")\n    sess = sesion(onnx)\n    t = time.time()\n    val = {c: puntajes(sess, ficha, a.validacion / c) for c in TIPOS if (a.validacion / c).is_dir()}\n    if not len(val.get(\"orig\", [])):\n        raise SystemExit(\"Faltan originales en la validación (carpeta orig/).\")\n    print(f\"Validación puntuada en {time.time() - t:.0f} s: \"\n          + \", \".join(f\"{c} {len(v)}\" for c, v in val.items()), flush=True)\n\n    anterior = float(ficha[\"umbral\"])\n    opciones = {}\n    for f in (.01, .015, .02, .03):\n        u = umbral_para(val[\"orig\"], f)\n        opciones[f\"{f:.1%}\"] = {\"umbral\": u, **tabla(val, u)}\n    nuevo = umbral_para(val[\"orig\"], a.fpr)\n    print(\"\\nEn validación (fotos nunca vistas), según la meta de falsas alarmas:\")\n    print(json.dumps(opciones, indent=2, ensure_ascii=False))\n\n    resultado = {\"meta_falsas_alarmas\": a.fpr, \"umbral_anterior\": anterior, \"umbral_nuevo\": nuevo,\n                 \"originales_para_calibrar\": int(len(val[\"orig\"])), \"validacion\": tabla(val, nuevo),\n                 \"opciones\": opciones}\n    if a.prueba:\n        prueba = {c: puntajes(sess, ficha, a.prueba / c) for c in TIPOS if (a.prueba / c).is_dir()}\n        resultado[\"prueba_umbral_nuevo\"] = tabla(prueba, nuevo)\n        resultado[\"prueba_umbral_anterior\"] = tabla(prueba, anterior)\n        print(\"\\nPrueba final (partición testing), umbral NUEVO vs ANTERIOR:\")\n        print(json.dumps({\"nuevo\": resultado[\"prueba_umbral_nuevo\"],\n                          \"anterior\": resultado[\"prueba_umbral_anterior\"]}, indent=2, ensure_ascii=False))\n\n    a.salida.mkdir(parents=True, exist_ok=True)\n    for f in (\"evidex_ia.onnx\", \"evidex_ia.safetensors\"):\n        if (a.modelo / f).exists() and (a.modelo / f).resolve() != (a.salida / f).resolve():\n            shutil.copyfile(a.modelo / f, a.salida / f)\n    ficha[\"umbral\"] = nuevo\n    ficha[\"calibracion\"] = resultado\n    ficha[\"fpr_objetivo\"] = a.fpr\n    (a.salida / \"evidex_ia.json\").write_text(json.dumps(ficha, indent=2, ensure_ascii=False), encoding=\"utf-8\")\n    print(f\"\\nListo: umbral {anterior:.6f} -> {nuevo:.6f}. Ficha nueva en {a.salida / 'evidex_ia.json'}\")\n\n\nif __name__ == \"__main__\":\n    main()\n"
}
for nombre, contenido in ARCHIVOS.items():
    Path("training", nombre).write_text(contenido, encoding="utf-8")
modelo = glob.glob("/kaggle/input/**/evidex_ia.onnx", recursive=True) if os.path.isdir("/kaggle") else \
         glob.glob(os.environ.get("EVX_MODELO", "modelos") + "/evidex_ia.onnx")
if not modelo:
    raise SystemExit("No encontré evidex_ia.onnx: agregue el modelo de la vuelta 3 con «+ Add Input».")
MODELO = os.path.dirname(modelo[0])
print("Modelo:", MODELO)

In [ ]:
# 3. Bajar fotos que el modelo nunca vio: «validation» para calibrar y «testing» para medir
VAL, PRUEBA = f"{BASE}/tgif_val", f"{BASE}/tgif_prueba"
sh(f"{sys.executable} training/descargar_tgif.py --destino {VAL} --n {N_ORIG_VAL} --particion validation --carpetas orig")
sh(f"{sys.executable} training/descargar_tgif.py --destino {VAL} --n {N_EDIT_VAL} --particion validation --carpetas sd2-sp,sd2-fr,sdxl-fr,ps-sp")
sh(f"{sys.executable} training/descargar_tgif.py --destino {PRUEBA} --n {N_PRUEBA} --particion testing --carpetas orig,sd2-sp,sd2-fr,sdxl-fr,ps-sp")

In [ ]:
# 4. Calibrar y medir (imprime las opciones de 1 %, 1,5 %, 2 % y 3 % y el resultado final)
sh(f"{sys.executable} training/calibrar.py --modelo {MODELO} --validacion {VAL} --prueba {PRUEBA} --fpr {FPR} --salida {SALIDA}")

In [ ]:
# 5. Empaquetar para descargar (Output → evidex_modelo_calibrado.zip)
shutil.make_archive(os.path.join(os.path.dirname(SALIDA) or ".", "evidex_modelo_calibrado"), "zip", SALIDA)
print("Listo:", sorted(os.listdir(SALIDA)))